# [N조] 데이터 분석 노트북 (2차시, 40분)

**이 노트북이 우리 조의 최종 산출물입니다.** 끝나면 [파일] → [다운로드] → .ipynb 로 받아 우리 조 공유 폴더에 올립니다.

| 시간 | 할 일 |
|---|---|
| 0~6분 | 준비 + 우리 조 설정 + [12·13차시] (**그대로 실행**, 메모만 채우기) |
| 6~13분 | [14차시] 결측값 |
| 13~22분 | [15차시] 새 열 · 그룹별 분석 · 함수 |
| 22~28분 | [16차시] 그래프 |
| 28~38분 | [17차시] 해석 + 사이트 ‘분석 결과 정리’ |

- 한 화면을 같이 보며 **의논해서** 채웁니다. 키보드는 [차시]마다 돌아가며 잡아요.
- 위에서부터 한 칸씩 ▶ 실행합니다. `\_\_\_\_`(밑줄 네 개)가 있는 곳만 **직접 채워야** 실행돼요.
- 막히면: 학습지 → 사이트 [코드] 탭 힌트 → 오류 해독기 → AI 코치(팀당 10번). 시간이 모자라면 **[14차시] 메모와 [15차시] 함수는 줄여도 돼요**(선생님 안내).

## 0. 준비와 우리 조 설정
사이트 [기획]에서 정한 것을 **아래 세 줄에만** 적습니다. 다음 칸들은 모두 이 변수를 써서 열 이름을 다시 쓸 필요가 없어요.

| 변수 | 뜻 | 예 |
|---|---|---|
| `DATA` | 데이터 파일 이름 | `"students"` |
| `GROUP` | 묶는 기준 열 | `"study_level"` |
| `VALUE` | 계산할 열 | `"final_score"` |

In [ ]:
# 처음 한 번만 실행 (한글 그래프용 도구 설치 포함, 1분 정도 걸려요)
!pip install -q koreanize-matplotlib
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib

# 우리 조 설정 — 따옴표 안에 이름만 써요 (열 이름은 print(df.columns)에서 복사)
DATA = "students"       # 데이터 파일 이름
GROUP = "study_level"      # 묶는 기준 열
VALUE = "final_score"      # 계산할 열

url = f"https://raw.githubusercontent.com/daradarav86/bigdata/main/data/{DATA}.csv"
df = pd.read_csv(url)
display(df.head())

---
# [12차시] 데이터 구조 확인  ·  *그대로 실행*
`head()`, `shape`, `columns`, `dtypes`, `info()` 를 한꺼번에 실행해 봅니다.

In [ ]:
# 그대로 실행하고, 아래 메모를 채워요 (이미 12차시에 배운 도구예요)
display(df.tail(3))
print(df.shape)       # (행 수, 열 수)
print(df.columns)     # 열 이름
print(df.dtypes)      # 열마다 자료형 (int·float = 숫자, object = 글자)
df.info()             # 비어 있지 않은 값의 개수

**메모 (12차시)**
- 한 행의 뜻: \_\_\_\_
- 값이 덜 찬 열(info의 non-null이 행 수보다 작은 열): \_\_\_\_

---
# [13차시] 통계 요약  ·  *그대로 실행*
`describe()`, `mean()`, `median()`, `value_counts()`

In [ ]:
# 그대로 실행하고, 아래 메모를 채워요
print(df[VALUE].describe())
print("평균:", round(df[VALUE].mean(), 2), "/ 중앙값:", round(df[VALUE].median(), 2))
print(df[GROUP].value_counts().head(10))

**메모 (13차시)**
- 평균 \_\_\_\_, 중앙값 \_\_\_\_ → 비슷한가요? 다르다면 왜일까요? \_\_\_\_
- 기준열에서 가장 많은 값 / 가장 적은 값: \_\_\_\_

---
# [14차시] 결측값 확인과 처리
원본 `df`는 그대로 두고, **제거한 표 `df_removed`** 와 **대체한 표 `df_filled`** 를 따로 만들어 비교합니다.

**hint:** `isnull().sum()`, `dropna()`, `copy()`, `fillna(값)`, `shape` — 이상한 값이 있으면 평균보다 **중앙값**으로 채워요.

In [ ]:
# 1. 열별 결측값 개수
print(df.isnull().sum())

# 2. 결측값이 있는 행을 제거한 표
df_removed = df.dropna()

# 3. 계산할열의 결측값을 대체한 표 (평균 mean 또는 중앙값 median)
df_filled = df.copy()
df_filled[VALUE] = df_filled[VALUE].fillna(df[VALUE].median())

# 4. 세 표 비교
print("행 수 — 원본:", df.shape[0], "/ 제거:", df_removed.shape[0], "/ 대체:", df_filled.shape[0])
print("평균 — 원본:", round(df[VALUE].mean(), 2), "/ 제거:", round(df_removed[VALUE].mean(), 2), "/ 대체:", round(df_filled[VALUE].mean(), 2))

**메모 (14차시)**
- 결측값이 있는 열과 개수: \_\_\_\_
- 우리가 고른 방법(그대로 두기 / 제거 / 대체)과 이유: \_\_\_\_

---
# [15차시] 새 열 만들기와 그룹별 분석
**hint:** `df["새열"] = df["열1"] - df["열2"]`, `groupby(GROUP)[VALUE].통계함수()` — 합계 `sum` / 평균 `mean` / 개수 `count`

In [ ]:
# 1. 새 열 만들기 — 사이트 [기획] 분석 계획에서 정한 새 열 (열끼리 + - * / 로 계산)
df["score_change"] = df["final_score"] - df["score_1"]
display(df.head())

In [ ]:
# 2. 기준열로 묶어서 합계·개수 비교
print(df.groupby(GROUP)[VALUE].sum())   # 합계
print(df.groupby(GROUP)[VALUE].count())   # 개수 — 그룹마다 몇 명?

# 3. 우리 질문의 답: 우리 통계함수 결과를 result에 저장
result = df.groupby(GROUP)[VALUE].mean()
print(result)

## [14·15차시] 함수로 만들기
같은 계산을 다른 기준열로도 해 보고 싶을 때 함수로 만들어 다시 불러요. **hint:** `def 함수이름(매개변수):` / `return 값`

In [ ]:
def group_mean(data, group_col):
    out = data.groupby(group_col)[VALUE].mean()
    return out

print(group_mean(df, GROUP))        # 우리 기준열
print(group_mean(df, "higher_education"))      # 다른 기준열로도 불러 보기

**메모 (15차시)**
- 만든 새 열과 그 뜻: \_\_\_\_
- 합계·개수·평균 중 우리 질문에 맞는 것과 이유: \_\_\_\_

---
# [16차시] 그래프
그룹 비교는 **막대 `plt.bar`**, 연도·월처럼 순서가 있으면 **선 `plt.plot`**. x축은 `result.index`, y축은 `result.values` (괄호 없음)

In [ ]:
plt.bar(result.index, result.values)
plt.title("공부 시간 구간별 최종 성적 평균")
plt.xlabel("공부 시간 구간")
plt.ylabel("최종 성적 평균 (0~20점)")
plt.show()

그래프를 캡처해서(Windows: `Win + Shift + S`) 사이트 [코드] 탭 **‘분석 결과 정리’** 의 그래프 칸에 붙여 넣어요.

---
# [17차시] 결과 읽기와 해석
**hint:** 가장 큰 그룹 `idxmax()` · 값 `max()`, 가장 작은 그룹 `idxmin()` · 값 `min()`

In [ ]:
print("가장 큰 그룹:", result.idxmax(), round(result.max(), 2))
print("가장 작은 그룹:", result.idxmin(), round(result.min(), 2))
print("차이:", round(result.max() - result.min(), 2), "/ 몇 배:", round(result.max() / result.min(), 2))

# 결측값을 제거한 표로도 결과가 비슷한지 확인
print(df_removed.groupby(GROUP)[VALUE].mean())

### 해석 (사이트 ‘분석 결과 정리’와 발표 5장에 옮겨요)
① **사실**: \_\_\_\_별 \_\_\_\_의 \_\_\_\_은 \_\_\_\_이(가) 가장 높고(\_\_\_\_), \_\_\_\_이(가) 가장 낮았다(\_\_\_\_).

② **비교**: 두 집단의 차이는 \_\_\_\_로, 약 \_\_\_\_배였다.

③ **의미**: 이는 \_\_\_\_와(과) \_\_\_\_가 함께 움직이는 경향이 있음을 보여 준다. *(원인이라고 단정하지 않기)*

**한계**: 다만 \_\_\_\_ 때문에 이 결과를 모든 대상에게 일반화하기 어렵다.

**진로 연결**: \_\_\_\_

## 제출 전 체크리스트 (12~17차시 필수 도구)
- [ ] [12] head/tail, shape, columns, dtypes, info()
- [ ] [13] describe(), 평균·중앙값 비교, value_counts()
- [ ] [14] isnull().sum(), df_removed와 df_filled 비교, 고른 방법과 이유
- [ ] [15] 새 열 1개, groupby로 sum·count·평균 비교
- [ ] [14·15] 함수(def, return)를 만들어 두 번 이상 불렀다
- [ ] [16] 제목·축 이름이 있는 그래프 1개
- [ ] [17] 가장 큰·작은 그룹과 차이, 해석 ①②③ + 한계
- [ ] 사이트 ‘분석 결과 정리’에 숫자·그래프·해석을 옮겼다
- [ ] [파일] → [다운로드] → .ipynb → 우리 조 공유 폴더에 올렸다

## 막히면?
| 증상 | 확인할 것 |
|---|---|
| `KeyError: '열이름'` | 열 이름 오타. `print(df.columns)`에서 복사해 붙여 넣기 |
| `NameError` | 위 셀을 먼저 실행했는지 (특히 0. 준비 셀, `result`를 만드는 셀) |
| 결과가 `NaN`이거나 비어 있음 | 계산할열이 숫자 열인지(`df.dtypes`), 결측값이 너무 많지 않은지(`isnull().sum()`) |
| 그래프 한글이 □로 깨짐 | 0. 준비 셀을 실행했는지. 그래도 깨지면 [런타임] → [세션 다시 시작] 후 처음부터 |
| 막대가 너무 많음 | `result.head(10)`으로 앞 10개만 그리기 |